# USPTO-50k mode / diversity analysis

Reproducible analysis of the pre-computed samples at
`/home/laabidn1/laabidn1/DiffAlign/experiments/align_absorbing_20260407_144212/` — 5000 USPTO-50k test products × 100 samples each, generated at **100 diffusion steps** from the epoch760 checkpoint.

Question answered here: **is the deployed checkpoint the source of the low-diversity complaint in the web app, or is it the sampling config (`diffusion_steps_eval=1`)?**

Running end-to-end takes ~2 min — the slow part is RDKit canonicalising ~500k SMILES. Outputs get cached to `uspto_analysis_out/` so reruns are instant.

In [ ]:
import os, re
from collections import Counter
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from rdkit import Chem, RDLogger
from rdkit.Chem import AllChem, DataStructs
RDLogger.DisableLog('rdApp.*')

ART_DIR = Path('/home/laabidn1/laabidn1/DiffAlign/experiments/align_absorbing_20260407_144212')
OUT_DIR = Path('uspto_analysis_out').resolve()
OUT_DIR.mkdir(exist_ok=True)

SAMPLE_FILES = sorted(ART_DIR.glob('samples_epoch760_steps100_cond500_sampercond100_s*.txt'))
print(f'Found {len(SAMPLE_FILES)} sample files. Output dir: {OUT_DIR}')

## Parsing helpers

Sample file format, one shard:
```
(cond 0) <true_reactants>>><true_product>:
\t<sample_0_reactants>>><sample_0_product>
\t<sample_1_reactants>>><sample_1_product>
... (100 samples per cond)
(cond 1) ...:
...
```

In [ ]:
HEADER_RE = re.compile(r'^\(cond (\d+)\) (.+):\s*$')

def parse_shard(path):
    """Yield (cond_idx_within_shard, true_rxn, [sample_rxn, ...])."""
    cond_idx, true_rxn, samples = None, None, []
    with open(path) as f:
        for raw in f:
            m = HEADER_RE.match(raw)
            if m:
                if cond_idx is not None:
                    yield cond_idx, true_rxn, samples
                cond_idx, true_rxn, samples = int(m.group(1)), m.group(2), []
            elif raw.startswith('\t'):
                samples.append(raw.strip())
    if cond_idx is not None:
        yield cond_idx, true_rxn, samples

def split_rxn(rxn):
    if '>>' not in rxn:
        return None, None
    rcts, prod = rxn.split('>>', 1)
    return rcts, prod

def canon(smi):
    """Sort fragments + canonicalise. Returns None if any fragment fails RDKit parse."""
    parts = smi.split('.')
    mols = [Chem.MolFromSmiles(p) for p in parts]
    if any(m is None for m in mols):
        return None
    return '.'.join(sorted(Chem.MolToSmiles(m, canonical=True) for m in mols))

def morgan_fp(smi, radius=2, nbits=2048):
    m = Chem.MolFromSmiles(smi)
    return AllChem.GetMorganFingerprintAsBitVect(m, radius, nbits) if m is not None else None

def pairwise_tanimoto(smiles_iter):
    fps = [morgan_fp(s) for s in smiles_iter]
    fps = [fp for fp in fps if fp is not None]
    if len(fps) < 2:
        return np.nan
    sims = [DataStructs.TanimotoSimilarity(fps[i], fps[j])
            for i in range(len(fps)) for j in range(i+1, len(fps))]
    return float(np.mean(sims))

def product_complexity(prod_smi):
    m = Chem.MolFromSmiles(prod_smi)
    if m is None:
        return {'heavy_atoms': np.nan, 'rings': np.nan, 'stereocenters': np.nan}
    return {
        'heavy_atoms': m.GetNumHeavyAtoms(),
        'rings': m.GetRingInfo().NumRings(),
        'stereocenters': len(Chem.FindMolChiralCenters(m, includeUnassigned=True, useLegacyImplementation=False)),
    }

## Parse + compute per-condition metrics

~2 min on first run. Reruns read the cached CSV in `uspto_analysis_out/per_condition.csv` — delete that file to force a re-parse.

In [ ]:
PER_COND_CSV = OUT_DIR / 'per_condition.csv'
MODE_REUSE_CSV = OUT_DIR / 'mode_reuse.csv'

if PER_COND_CSV.exists() and MODE_REUSE_CSV.exists():
    df = pd.read_csv(PER_COND_CSV)
    mr_df = pd.read_csv(MODE_REUSE_CSV)
    print(f'Loaded cached CSVs — {len(df)} conditions.')
else:
    rows, top_modes = [], []
    for shard in SAMPLE_FILES:
        print(f'shard {shard.name} ...', flush=True)
        shard_start = int(re.search(r'_s(\d+)\.txt$', shard.name).group(1))
        for cond_idx, true_rxn, samples in parse_shard(shard):
            true_rcts_raw, true_prod_raw = split_rxn(true_rxn)
            true_rcts_canon = canon(true_rcts_raw) if true_rcts_raw else None
            true_prod_canon = canon(true_prod_raw) if true_prod_raw else None

            sample_rcts_canon, n_invalid, n_identity = [], 0, 0
            for s in samples:
                rcts_raw, prod_raw = split_rxn(s)
                if rcts_raw is None:
                    n_invalid += 1; continue
                rcts_c = canon(rcts_raw)
                if rcts_c is None:
                    n_invalid += 1; continue
                sample_rcts_canon.append(rcts_c)
                prod_c = canon(prod_raw)
                if prod_c is not None and rcts_c == prod_c:
                    n_identity += 1

            counter = Counter(sample_rcts_canon)
            unique_modes = list(counter.keys())
            top_mode = counter.most_common(1)[0] if counter else (None, 0)
            mean_tan = pairwise_tanimoto(unique_modes[:50]) if len(unique_modes) > 1 else np.nan
            cx = product_complexity(true_prod_canon) if true_prod_canon else {'heavy_atoms': np.nan, 'rings': np.nan, 'stereocenters': np.nan}

            rows.append({
                'cond':              shard_start + cond_idx,
                'true_product':      true_prod_canon,
                'true_reactants':    true_rcts_canon,
                'n_samples':         len(samples),
                'n_valid':           len(sample_rcts_canon),
                'n_invalid':         n_invalid,
                'n_unique':          len(unique_modes),
                'top_mode_count':    top_mode[1],
                'top_mode_smiles':   top_mode[0],
                'mean_tanimoto':     mean_tan,
                'identity_count':    n_identity,
                'truth_in_modes':    bool(true_rcts_canon and true_rcts_canon in counter),
                'truth_is_top':      bool(true_rcts_canon and top_mode[0] == true_rcts_canon),
                'truth_count':       counter.get(true_rcts_canon, 0) if true_rcts_canon else 0,
                **cx,
            })
            if top_mode[0] is not None and true_prod_canon is not None:
                top_modes.append((top_mode[0], true_prod_canon))

    df = pd.DataFrame(rows)
    df.to_csv(PER_COND_CSV, index=False)

    mode_to_products = {}
    for top_smi, prod in top_modes:
        mode_to_products.setdefault(top_smi, set()).add(prod)
    mr_df = pd.DataFrame(
        sorted(({'top_mode_smiles': k, 'n_distinct_products': len(v)} for k, v in mode_to_products.items()),
               key=lambda r: r['n_distinct_products'], reverse=True)
    )
    mr_df.to_csv(MODE_REUSE_CSV, index=False)
    print(f'Processed {len(df)} conditions. Wrote {PER_COND_CSV.name} and {MODE_REUSE_CSV.name}.')

df.head()

## Diversity buckets — the inpainting-relevance view

`n_unique / 100` is the key diversity metric. 1 = fully collapsed; 100 = chaotic. The 5–20 range is the inpainting sweet spot (multiple plausible disconnections the user can bias between).

In [ ]:
def bucket(n):
    if n <= 1:   return '1 (collapsed)'
    if n <= 4:   return '2-4 (low)'
    if n <= 20:  return '5-20 (sweet spot for inpainting)'
    if n <= 50:  return '21-50 (high)'
    return '51+ (very high)'

df['bucket'] = df['n_unique'].apply(bucket)
bucket_order = ['1 (collapsed)', '2-4 (low)', '5-20 (sweet spot for inpainting)', '21-50 (high)', '51+ (very high)']
bucket_counts = df['bucket'].value_counts().reindex(bucket_order, fill_value=0)
pd.DataFrame({'count': bucket_counts, 'pct': (bucket_counts / len(df) * 100).round(1)})

In [ ]:
# Finer histogram
bins = [0, 1, 2, 4, 10, 20, 30, 50, 80, 101]
labels = ['1', '2', '3-4', '5-10', '11-20', '21-30', '31-50', '51-80', '81-100']
hist = pd.cut(df['n_unique'], bins=bins, labels=labels, right=True).value_counts().reindex(labels, fill_value=0)

fig, ax = plt.subplots(figsize=(9, 4))
ax.bar(labels, hist.values, color='steelblue')
ax.set_xlabel('n_unique per condition (out of 100 samples)')
ax.set_ylabel('# conditions')
ax.set_title('Distribution of mode counts across 4951 USPTO-50k test conditions')
plt.tight_layout()

## Truth recovery — diversity vs. quality

Does diversity come at the cost of accuracy? Grouping by diversity bucket and looking at how often the ground-truth reactants appear.

In [ ]:
rows = []
for lab in labels:
    sub = df[pd.cut(df['n_unique'], bins=bins, labels=labels, right=True) == lab]
    if len(sub) == 0:
        continue
    rows.append({
        'bucket':        lab,
        'n':             len(sub),
        'truth_in_modes_%': round(sub['truth_in_modes'].mean() * 100, 1),
        'truth_is_top_%':  round(sub['truth_is_top'].mean() * 100, 1),
        'mean_tanimoto': round(sub['mean_tanimoto'].mean(), 3) if not sub['mean_tanimoto'].isna().all() else None,
    })
pd.DataFrame(rows)

## Collapsed-group examples

What does `n_unique == 1` look like? Most of these should be the model being *confident*, not broken — check `truth_is_top`.

In [ ]:
collapsed = df[df['n_unique'] == 1]
print(f'{len(collapsed)} collapsed conditions. '
      f'truth_is_top in {collapsed["truth_is_top"].sum()} ({collapsed["truth_is_top"].mean()*100:.1f}%).')

for _, r in collapsed.head(8).iterrows():
    tag = '✓ TRUTH' if r['truth_is_top'] else '✗ NOT truth'
    print(f"\ncond {int(r['cond'])}  heavy={int(r['heavy_atoms']) if pd.notna(r['heavy_atoms']) else '?'}  {tag}")
    print(f'  product: {r["true_product"][:90]}')
    print(f'  mode:    {r["top_mode_smiles"][:90]}')
    print(f'  truth:   {r["true_reactants"][:90] if pd.notna(r["true_reactants"]) else "(?)"}')

## Top-mode dominance

Even within a "diverse" condition, how concentrated is the mass on the top mode? `top_mode_count` is the number of the 100 samples that match the plurality answer.

In [ ]:
rows = []
for thr in [100, 90, 75, 50, 25, 10]:
    n = (df['top_mode_count'] >= thr).sum()
    rows.append({'top_mode_count >=': thr, 'n': int(n), 'pct': round(n / len(df) * 100, 1)})
pd.DataFrame(rows)

## Mode reuse — is the model genuinely product-conditional?

If the same top-mode SMILES appeared as the answer for many different products, the model would be falling back on a generic library. Mode reuse = 1 means "every top answer is unique to its product".

In [ ]:
print(f'Total distinct top-mode SMILES: {len(mr_df)}')
print(f'Top-mode SMILES appearing as top for >=2 products: {int((mr_df["n_distinct_products"] >= 2).sum())}')
print(f'Most-reused top mode: {mr_df.iloc[0]["n_distinct_products"]} products')
mr_df.head(5)

## Failure-mode counters — identity and validity

In [ ]:
print('Identity reactions (reactants == product):')
print(f'  mean / condition               : {df["identity_count"].mean():.2f}')
print(f'  conditions with 0 identity     : {int((df["identity_count"] == 0).sum())} ({(df["identity_count"] == 0).mean()*100:.1f}%)')
print(f'  conditions with >=10 identity  : {int((df["identity_count"] >= 10).sum())}')
print(f'  conditions with >=50 identity  : {int((df["identity_count"] >= 50).sum())}')
print()
print('Sample validity (RDKit-parseable):')
print(f'  mean valid / 100         : {df["n_valid"].mean():.1f}')
print(f'  conditions with <80 valid: {int((df["n_valid"] < 80).sum())}')
print(f'  conditions with <50 valid: {int((df["n_valid"] < 50).sum())}')

## Does diversity correlate with product complexity?

In [ ]:
rows = []
for col in ['heavy_atoms', 'rings', 'stereocenters']:
    sub = df[[col, 'n_unique']].dropna()
    if len(sub) > 1 and sub[col].std() > 0:
        rows.append({'feature': col, 'pearson_r_with_n_unique': round(sub[col].corr(sub['n_unique']), 3), 'n': len(sub)})
pd.DataFrame(rows)

## Findings

See `uspto_analysis_out/SUMMARY.md` for the full write-up. One-paragraph version:

The checkpoint at **100 diffusion steps is healthy** — 57.5% of products land in the 5–20 unique-mode inpainting sweet spot, truth-in-top-1 is 56.9%, mode reuse is 0 (every top answer is product-unique), identity-rxn and invalid-sample rates are both low. The deployed app's "low diversity" complaint is almost entirely attributable to `diffusion_steps_eval=1`, not to the checkpoint. **Collapsed cases (`n_unique=1`, 1.0%) are mostly correct** (~92% match ground truth) — confidence, not failure. The residual quality-vs-diversity question (are diverse outputs any good? is collapse overfitting or certainty?) is worth its own investigation on top of this data — a natural next notebook.